In [20]:
from datetime import datetime

from pytz import timezone

tz_ny = timezone("America/New_York")
news_start = tz_ny.localize(datetime(2024, 12, 4, 7, 30))
news_end = tz_ny.localize(datetime(2024, 12, 4, 9, 00))

In [21]:
from pytz import utc


def format_datetime_for_x(dt: datetime):
    """https://docs.x.com/x-api/posts/search-all-posts#parameter-start-time"""
    return dt.astimezone(utc).strftime("%Y-%m-%dT%H:%M:%SZ")

In [22]:
format_datetime_for_x(news_start)

'2024-12-04T12:30:00Z'

https://docs.x.com/x-api/posts/search-all-posts


In [46]:
import os

import httpx

token = os.environ["X_BEARER_TOKEN"]


response = httpx.get(
    "https://api.x.com/2/tweets/search/all",
    headers={"Authorization": f"Bearer {token}"},
    params={
        "query": '("brian thompson" OR unitedhealthcare ceo) -is:retweet',
        "start_time": format_datetime_for_x(news_start),
        "end_time": format_datetime_for_x(news_end),
        "max_results": 50,
        "sort_order": "recency",
        "expansions": "author_id",
        "tweet.fields": "created_at,author_id",
        "user.fields": "username",
    },
)
response.json()

{'data': [{'id': '1864308655396356447',
   'created_at': '2024-12-04T13:59:55.000Z',
   'edit_history_tweet_ids': ['1864308655396356447'],
   'text': '@zerohedge https://t.co/IihlrXHdDP',
   'author_id': '1589626443348299779'},
  {'id': '1864308567496642945',
   'created_at': '2024-12-04T13:59:34.000Z',
   'edit_history_tweet_ids': ['1864308567496642945'],
   'text': 'Exclusive | UnitedHealthcare CEO Brian Thompson fatally shot outside of Hilton hotel in NYC: sources https://t.co/4qkrib4a5z',
   'author_id': '15078874'},
  {'id': '1864308501239202033',
   'created_at': '2024-12-04T13:59:18.000Z',
   'edit_history_tweet_ids': ['1864308501239202033'],
   'text': '$UNH | According to PIX11, UnitedHealthcare CEO Brian Thompson has reportedly been fatally shot in Midtown Manhattan. \n\nNote: Thompson is the CEO of UnitedHealthcare, a division of UnitedHealth Group, not the CEO of UnitedHealth Group itself.',
   'author_id': '1508426880860704771'},
  {'id': '1864308492699320568',
   'created

In [60]:
def make_clickable(val):
    # target _blank to open new window
    return '<a target="_blank" href="{}">{}</a>'.format(val, val)

In [70]:
import pandas as pd

payload = response.json()
tweets = pd.DataFrame(payload["data"])
users = pd.DataFrame(payload["includes"]["users"])

tweets = tweets.merge(
    users[["id", "username", "name"]],
    left_on="author_id",
    right_on="id",
    how="left",
    suffixes=("", "_user"),
)

tweets = tweets.assign(
    created_at=pd.to_datetime(tweets["created_at"]).dt.tz_convert(tz_ny),
    # derive URL
    url="https://x.com/" + tweets["username"].fillna("i") + "/status/" + tweets["id"],
)

tweets = tweets[
    [
        "created_at",
        "username",
        "name",
        "text",
        "url",
    ]
]
tweets = tweets.sort_values("created_at")

tweets.style.format({"url": make_clickable})

,created_at,username,name,text,url
44,2024-12-04 08:54:04-05:00,ThilakMSD07,THILAK DAS,@nypmetro Brian Thompson,https://x.com/ThilakMSD07/status/1864307183086915932
43,2024-12-04 08:54:06-05:00,faststocknewss,FSMN,*CEO OF UNITEDHEALTHCARE FATALLY SHOT IN MANHATTAN: PIX 11 $UNH,https://x.com/faststocknewss/status/1864307192847061103
42,2024-12-04 08:54:17-05:00,nypost,New York Post,CEO of UnitedHealthcare fatally shot outside of Hilton hotel in Midtown in possible targeted attack: sources https://t.co/HCvBXdP80G https://t.co/i2x8rPxfpC,https://x.com/nypost/status/1864307237381882132
41,2024-12-04 08:54:21-05:00,DanMannarino,Dan Mannarino,"NEW: Multiple sources confirm to @PIX11News that United Healthcare CEO Brian Thompson was shot and killed outside the Hilton hotel in midtown just before 7am, where he was slated to speak at a investor meeting later today. @PIX11News",https://x.com/DanMannarino/status/1864307254733705559
40,2024-12-04 08:54:25-05:00,tradfi,tradfi news,UNITEDHEALTH CEO BRIAN THOMPSON FATALLY SHOT IN MIDTOWN: PIX11 $UNH,https://x.com/tradfi/status/1864307273432224234
39,2024-12-04 08:54:41-05:00,michaelgmcquaid,McQuaid,CEO OF UNITEDHEALTHCARE FATALLY SHOT IN MANHATTAN: PIX 11 UNITEDHEALTH SHARES FALL 2.7% PREMARKET36 Holy shit,https://x.com/michaelgmcquaid/status/1864307338141929670
38,2024-12-04 08:54:50-05:00,Robertajair22,Roberta🇧🇷,"*CEO DA UNITEDHEALTHCARE FOI MORTO A TIRO EM MANHATTAN: PIX 11 *AÇÕES DA UNITEDHEALTH CAEM 2,7% NO PRÉ-MERCADO 👀 https://t.co/AZd7yyY9p6",https://x.com/Robertajair22/status/1864307379027722313
37,2024-12-04 08:54:51-05:00,Zerodonotblock,Mr. Floppy 📈📉,MAGA SCHWURBLER MACHEN ERNST. CEO OF UNITEDHEALTHCARE FATALLY SHOT IN MANHATTAN: PIX 11,https://x.com/Zerodonotblock/status/1864307379451371999
36,2024-12-04 08:54:52-05:00,Arturojrobles,Arturo Robles,CEO OF UNITEDHEALTHCARE FATALLY SHOT IN MANHATTAN: PIX 11 $UNH,https://x.com/Arturojrobles/status/1864307386749472866
35,2024-12-04 08:54:53-05:00,adar170,mets17,*CEO OF UNITEDHEALTHCARE FATALLY SHOT IN MANHATTAN: PIX 11,https://x.com/adar170/status/1864307388129456147
